# Analyse des Risques Environnementaux - Réseau Ferroviaire SNCF

Ce projet réalise une analyse quantitative des risques environnementaux impactant le Réseau Ferré National (RFN). 

**Objectif :** Identifier les tronçons vulnérables, quantifier le risque et proposer des stratégies d'adaptation.

## 1. Visualisation du Réseau Ferroviaire
L'objectif est d'abord de cartographier l'ensemble du réseau pour disposer d'une base géométrique fiable.
**Sources de données :**
- Géométrie et Vitesses : API SNCF Open Data

In [ ]:
import folium
import requests
import pandas as pd
import logging

# 1. Initialisation de la Carte
sncf_map = folium.Map(
    location=[46.603354, 1.888334], zoom_start=6, tiles='CartoDB positron', height=800
)

URL_SHAPES = "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/formes-des-lignes-du-rfn/exports/geojson?limit=-1"
URL_SPEED = "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/vitesse-maximale-nominale-sur-ligne/exports/geojson?limit=-1"

def fetch_geojson(url):
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        logging.error(f"Erreur lors de la récupération des données depuis {url}: {e}")
        return None

shapes_data = fetch_geojson(URL_SHAPES)
speed_data = fetch_geojson(URL_SPEED)

if shapes_data and speed_data:
    def get_key(props):
        return f"{props.get('code_ligne')}_{props.get('rg_troncon')}"
    
    final_features = []
    existing_keys = set()
    
    for feature in speed_data.get('features', []):
        props = feature.get('properties', {})
        key = get_key(props)
        existing_keys.add(key)
        standard_props = {
            'Code ligne': props.get('code_ligne', 'Inconnu'),
            'Status': props.get('lib_ligne', props.get('libelle', 'Inconnu')),
            'Vitesse max': props.get('v_max', 'N/A'),
            'Source': 'Vitesse',
        }
        feature['properties'] = {**props, **standard_props}
        final_features.append(feature)
    
    for feature in shapes_data.get('features', []):
        props = feature.get('properties', {})
        key = get_key(props)
        if key not in existing_keys:
            standard_props = {
                'Code ligne': props.get('code_ligne', 'Inconnu'),
                'Status': props.get('libelle', 'Inconnu'),
                'Vitesse max': 'N/A',
                'Source': 'Formes',
            }
            feature['properties'] = {**props, **standard_props}
            final_features.append(feature)
    
    geojson_combined = {'type': 'FeatureCollection', 'features': final_features}
    
    def style_speed(feature):
        vmax = feature['properties'].get('v_max', 'N/A')
        try:
            val = float(vmax)
            if val >= 220:
                color = 'red'
            elif val >= 160:
                color = 'orange'
            elif val >= 80:
                color = 'yellow'
            else:
                color = 'green'
        except (ValueError, TypeError):
            color = 'gray'
        return {'color': color, 'weight': 2, 'opacity': 0.8}
    
    folium.GeoJson(
        geojson_combined,
        name='Réseau SNCF',
        style_function=style_speed,
        tooltip=folium.GeoJsonTooltip(fields=['Code ligne', 'Status', 'Vitesse max']),
    ).add_to(sncf_map)
    folium.LayerControl().add_to(sncf_map)
    logging.info("Carte SNCF générée avec succès.")
else:
    logging.error("Erreur de chargement.")

sncf_map

## 2. Analyse des Jours Chauds (>30°C) - 2025-2026

Cette section analyse les données de température pour identifier les zones géographiques avec le plus grand nombre de jours où la température a dépassé 30°C.

**Source** : Données Météo-France 2025-2026

In [ ]:
import pandas as pd
import folium
import numpy as np
from scipy.spatial import KDTree

print("Chargement des données de température...")
temp_df = pd.read_csv('data/temperature/temperature_2025_2026.csv', sep=';')
print(f"Données chargées: {len(temp_df):,} enregistrements")

hot_days = temp_df[temp_df['TX'] > 30].copy()
station_hot_days = hot_days.groupby(['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON']).size().reset_index(name='hot_days_count')

all_stations = temp_df.groupby(['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON']).size().reset_index(name='total_days')
station_stats = all_stations.merge(station_hot_days, on=['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON'], how='left')
station_stats['hot_days_count'] = station_stats['hot_days_count'].fillna(0).astype(int)

print(f"Total stations: {len(station_stats)}")

def get_color(count):
    if count >= 50:
        return "#BB0303"
    elif count >= 30:
        return '#ff8c00'
    elif count >= 20:
        return '#ffff00'
    elif count >= 10:
        return '#00ff00'
    elif count >= 5:
        return '#00ffff'
    else:
        return '#add8e6'

print("Calcul des rayons optimisés pour couvrir tout le territoire...")
coords = np.array([[row['LAT'], row['LON']] for idx, row in station_stats.iterrows()])
tree = KDTree(coords)

radii_km = []
for i, coord in enumerate(coords):
    distances, _ = tree.query(coord, k=4)
    
    if len(distances) > 1:
        avg_distance_deg = np.mean(distances[1:3])
        distance_km = avg_distance_deg * 111
        
        if len(distances) > 3 and distances[3] < 0.5:
            radii_km.append(distance_km * 0.45)
        else:
            radii_km.append(distance_km * 0.55)
    else:
        radii_km.append(60)

print(f"Rayons optimisés: moyenne={np.mean(radii_km):.1f} km, max={max(radii_km):.1f} km")
print(f"Distribution: min={np.min(radii_km):.1f} km, médiane={np.median(radii_km):.1f} km")

heat_map = folium.Map(
    location=[46.603354, 1.888334],
    zoom_start=6,
    tiles='CartoDB positron',
    height=800
)

print("Ajout des cercles optimisés...")
for i, row in station_stats.iterrows():
    folium.Circle(
        location=[row['LAT'], row['LON']],
        radius=int(radii_km[i] * 1000),
        color=get_color(row['hot_days_count']),
        fill=True,
        fill_color=get_color(row['hot_days_count']),
        fill_opacity=0.7,
        weight=1,
        popup=f"""
        <b>{row['NOM_USUEL']}</b><br>
        Jours > 30°C: {row['hot_days_count']}<br>
        Station: {row['NUM_POSTE']}<br>
        Rayon: {radii_km[i]:.0f} km
        """,
        tooltip=f"{row['NOM_USUEL']}: {row['hot_days_count']} jours"
    ).add_to(heat_map)

legend_html = """
<div style="position: fixed;
     top: 50px; right: 50px; width: 220px; height: 220px;
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white;
     padding: 12px;
     ">
     <b>Jours > 30°C (2025-2026)</b><br>
     <hr>
     <i style="background:#add8e6; width:20px; height:20px; display:inline-block; border:1px solid grey;"></i> <5 jours<br>
     <i style="background:#00ffff; width:20px; height:20px; display:inline-block; border:1px solid grey;"></i> 5-10 jours<br>
     <i style="background:#00ff00; width:20px; height:20px; display:inline-block; border:1px solid grey;"></i> 10-20 jours<br>
     <i style="background:#ffff00; width:20px; height:20px; display:inline-block; border:1px solid grey;"></i> 20-30 jours<br>
     <i style="background:#ff8c00; width:20px; height:20px; display:inline-block; border:1px solid grey;"></i> 30-50 jours<br>
     <i style="background:#BB0303; width:20px; height:20px; display:inline-block; border:1px solid grey;"></i> 50+ jours<br>
</div>
"""

heat_map.get_root().html.add_child(folium.Element(legend_html))

print("Carte générée avec succès!")
print(f"Couverture optimisée avec {len(station_stats)} cercles")
print(f"Rayon moyen: {np.mean(radii_km):.1f} km")
heat_map

## 3. Analyse du Risque Thermique sur le Réseau SNCF

Cette section calcule un coefficient de risque thermique pour les lignes ferroviaires, combinant:
- Vitesse maximale des lignes (plus rapide = plus sensible à la chaleur)
- Nombre de jours > 30°C dans les zones traversées

**Méthode**: Les lignes rapides sont plus vulnérables à la dilatation thermique et aux déformations.

In [ ]:
import pandas as pd
import folium
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

print("Calcul du risque thermique pour le réseau SNCF...")

# Charger les données
temp_df = pd.read_csv('data/temperature/temperature_2025_2026.csv', sep=';')
hot_days = temp_df[temp_df['TX'] > 30]
station_hot_days = hot_days.groupby(['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON']).size().reset_index(name='hot_days_count')

# Créer GeoDataFrame des stations
points = [Point(row['LON'], row['LAT']) for idx, row in station_hot_days.iterrows()]
station_gdf = gpd.GeoDataFrame(station_hot_days, geometry=points, crs='EPSG:4326')

# Charger données SNCF avec vitesses (réutiliser la logique de la première cellule)
import requests
URL_SHAPES = "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/formes-des-lignes-du-rfn/exports/geojson?limit=-1"
URL_SPEED = "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/vitesse-maximale-nominale-sur-ligne/exports/geojson?limit=-1"

def fetch_geojson(url):
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Erreur lors de la récupération des données depuis {url}: {e}")
        return None

shapes_data = fetch_geojson(URL_SHAPES)
speed_data = fetch_geojson(URL_SPEED)

if shapes_data and speed_data:
    def get_key(props):
        return f"{props.get('code_ligne')}_{props.get('rg_troncon')}"
    
    final_features = []
    existing_keys = set()
    
    for feature in speed_data.get('features', []):
        props = feature.get('properties', {})
        key = get_key(props)
        existing_keys.add(key)
        standard_props = {
            'Code ligne': props.get('code_ligne', 'Inconnu'),
            'Status': props.get('lib_ligne', props.get('libelle', 'Inconnu')),
            'Vitesse max': props.get('v_max', 'N/A'),
            'Source': 'Vitesse',
        }
        feature['properties'] = {**props, **standard_props}
        final_features.append(feature)
    
    for feature in shapes_data.get('features', []):
        props = feature.get('properties', {})
        key = get_key(props)
        if key not in existing_keys:
            standard_props = {
                'Code ligne': props.get('code_ligne', 'Inconnu'),
                'Status': props.get('libelle', 'Inconnu'),
                'Vitesse max': 'N/A',
                'Source': 'Formes',
            }
            feature['properties'] = {**props, **standard_props}
            final_features.append(feature)
    
    geojson_combined = {'type': 'FeatureCollection', 'features': final_features}
    rail_lines = gpd.GeoDataFrame.from_features(geojson_combined['features'], crs='EPSG:4326')
else:
    print("Erreur de chargement des données SNCF")
    rail_lines = gpd.GeoDataFrame()

# Calculer risque thermique
def calculate_risk(line, stations):
    try:
        buffer_geom = line['geometry'].buffer(0.1)
        nearby = stations[stations.within(buffer_geom)]
        return nearby['hot_days_count'].mean() if len(nearby) > 0 else 0
    except:
        return 0

rail_lines['avg_hot_days'] = [calculate_risk(line, station_gdf) for idx, line in rail_lines.iterrows()]

# Classer par risque
def get_risk_category(days, vmax):
    if vmax <= 60:  # Lignes lentes = risque négligeable
        return 'Négligeable'
    elif days >= 60: return 'Extreme' 
    elif days >= 50: return 'Très Élevé'
    elif days >= 30: return 'Élevé'
    elif days >= 20: return 'Moyen'
    elif days >= 10: return 'Faible'
    else: return 'Négligeable'

# Extraire la vitesse max des propriétés - méthode corrigée pour GeoDataFrame
rail_lines['v_max'] = rail_lines['Vitesse max'].apply(
    lambda x: float(x) if str(x) != 'N/A' else 100
)

# Appliquer la classification
rail_lines['risk_category'] = rail_lines.apply(
    lambda row: get_risk_category(row['avg_hot_days'], row['v_max']), 
    axis=1
)

# Créer la carte
risk_map = folium.Map(location=[46.603354, 1.888334], zoom_start=6, height=800, tiles='CartoDB positron',)

# Style basé sur la catégorie de risque (et non sur les jours)
def style_risk(feature):
    category = feature['properties'].get('risk_category', 'Négligeable')
    if category == 'Extreme':
        return {'color': "#000000", 'weight': 3, 'opacity': 0.8}
    elif category == 'Très Élevé':
        return {'color': "#BB0303", 'weight': 3, 'opacity': 0.8}
    elif category == 'Élevé':
        return {'color': '#ff8c00', 'weight': 3, 'opacity': 0.8}
    elif category == 'Moyen':
        return {'color': '#ffff00', 'weight': 3, 'opacity': 0.8}
    elif category == 'Faible':
        return {'color': "#00ff00", 'weight': 3, 'opacity': 0.8}
    else:  # Négligeable
        return {'color': "#00ffff", 'weight': 3, 'opacity': 0.8}

folium.GeoJson(
    rail_lines,
    style_function=style_risk,
    tooltip=folium.GeoJsonTooltip(
        fields=['Code ligne', 'risk_category', 'avg_hot_days', 'Vitesse max'],
        aliases=['Ligne:', 'Risque:', 'Jours >30°C:', 'Vitesse max:']
    )
).add_to(risk_map)

# Légende
legend_html = """
<div style="position: fixed; top: 50px; right: 50px; width: 200px; height: 180px;
     border:2px solid grey; z-index:9999; font-size:14px; background-color:white; padding: 12px;">
     <b>Risque Thermique SNCF</b><br><hr>
     <i style="background:#00ffff; width:20px; height:20px; display:inline-block; border:1px solid grey;"></i> Négligeable (<10j ou <80km/h)<br>
     <i style="background:#00ff00; width:20px; height:20px; display:inline-block; border:1px solid grey;"></i> Faible (10-20j)<br>
     <i style="background:#ffff00; width:20px; height:20px; display:inline-block; border:1px solid grey;"></i> Moyen (20-30j)<br>
     <i style="background:#ff8c00; width:20px; height:20px; display:inline-block; border:1px solid grey;"></i> Élevé (40+j)<br>
     <i style="background:#bb0303; width:20px; height:20px; display:inline-block; border:1px solid grey;"></i> Très Élevé (50+j)<br>
</div>
"""

risk_map.get_root().html.add_child(folium.Element(legend_html))

print(f"Analyse terminée! Lignes: Élevé={(rail_lines['risk_category'] == 'Élevé').sum()}, Moyen={(rail_lines['risk_category'] == 'Moyen').sum()}, Faible={(rail_lines['risk_category'] == 'Faible').sum()}, Négligeable={(rail_lines['risk_category'] == 'Négligeable').sum()}")

risk_map

## 4. Retrait et Gonflement des Argiles (RGA)

In [ ]:
import geopandas as gpd
import folium

print("Chargement des zones à risque d'argile (version optimisée)...")

try:
    # Charger les données avec un filtre plus strict pour éviter les crashes
    argile_gdf = gpd.read_file('./data/AleaRG_2025_Fxx_L93/AleaRG_2025_Fxx_L93.shp')
    
    print(f"✓ Données brutes: {len(argile_gdf):,} enregistrements")
    
    # Filtrer uniquement les risques élevés (niveau 3.0 = Fort)
    if 'niveau' in argile_gdf.columns:
        argile_high_risk = argile_gdf[argile_gdf['niveau'] == 3.0]  # Seulement niveau 3
        print(f"✓ Zones à risque fort: {len(argile_high_risk):,} enregistrements")
    else:
        argile_high_risk = argile_gdf
        print("⚠ Colonne 'niveau' non trouvée")
    
    # Filtrer les plus grandes zones pour réduire la charge
    if 'surf_m2' in argile_high_risk.columns:
        # Garder les 5% plus grandes zones (au lieu de 10%)
        argile_large = argile_high_risk.nlargest(int(len(argile_high_risk)*0.05), 'surf_m2')
        print(f"✓ Plus grandes zones sélectionnées: {len(argile_large):,} enregistrements")
        print(f"   Surface minimale: {argile_large['surf_m2'].min():,.0f} m²")
    else:
        argile_large = argile_high_risk
        print("⚠ Colonne 'surf_m2' non trouvée")
    
    # Convertir en WGS84
    argile_large = argile_large.to_crs('EPSG:4326')
    
    # Créer la carte
    argile_map = folium.Map(
        location=[46.603354, 1.888334], 
        zoom_start=6,
        tiles='CartoDB positron'
    )
    
    # Style simple
    folium.GeoJson(
        argile_large,
        style_function=lambda x: {
            'fillColor': 'red',
            'color': 'darkred',
            'weight': 1,
            'fillOpacity': 0.5
        },
        tooltip=folium.GeoJsonTooltip(
            fields=['niveau', 'surf_m2'],
            aliases=['Risque:', 'Surface (m²):'],
            localize=True
        )
    ).add_to(argile_map)
    
    # Légende
    legend_html = """
    <div style="position: fixed; bottom: 50px; right: 50px; width: 180px; height: 80px;
         border:2px solid grey; z-index:9999; font-size:14px; background-color:white; padding: 10px;">
         <b>Zones à risque fort d'argile</b><br>
         <i style="background:red; width:15px; height:15px; display:inline-block; border:1px solid grey;"></i> Risque Fort (niveau 3)<br>
         <small>(5% des plus grandes zones)</small>
    </div>
    """
    
    argile_map.get_root().html.add_child(folium.Element(legend_html))
    
    print("\n✓ Carte générée avec succès!")
    print(f"   Affichage de {len(argile_large):,} zones à risque fort")
    
    # Afficher la carte
    argile_map
    
except Exception as e:
    print(f"\n✗ Erreur: {e}")
    print("\nSolutions:")
    print("1. Vérifiez que le fichier AleaRG_2025_Fxx_L93.shp existe bien")
    print("2. Essayez un filtre encore plus strict: nlargest(int(len()*0.02), 'surf_m2')")
    print("3. Redémarrez le kernel et relancez la cellule")
    import traceback
    traceback.print_exc()

In [ ]:
argile_map

In [ ]:
import geopandas as gpd
import folium
import requests
import pandas as pd

print("Analyse du risque argiles sur le réseau SNCF...")

# 1. Charger les données d'argile (5% plus grandes zones, tous niveaux)
print("\n1. Chargement des zones à risque d'argile...")
argile_gdf = gpd.read_file('./data/AleaRG_2025_Fxx_L93/AleaRG_2025_Fxx_L93.shp')

if 'niveau' in argile_gdf.columns and 'surf_m2' in argile_gdf.columns:
    # Filtrer les risques moyens et forts (niveau 2.0 et 3.0)
    argile_risk = argile_gdf[argile_gdf['niveau'].isin([2.0, 3.0])]
    print(f"   Zones à risque: {len(argile_risk):,} enregistrements")

    # Garder les 5% plus grandes zones
    argile_large = argile_risk.nlargest(int(len(argile_risk)*0.05), 'surf_m2')
    print(f"   5% plus grandes zones: {len(argile_large):,} enregistrements")
else:
    argile_large = argile_gdf
    print("   ⚠ Colonnes manquantes")

argile_large = argile_large.to_crs('EPSG:4326')

# 2. Charger les données SNCF
print("\n2. Chargement du réseau SNCF...")
URL_SHAPES = "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/formes-des-lignes-du-rfn/exports/geojson?limit=-1"
shapes_data = requests.get(URL_SHAPES, timeout=60).json()
rail_lines = gpd.GeoDataFrame.from_features(shapes_data['features'], crs='EPSG:4326')

# 3. Calculer le risque argile pour chaque ligne
print("\n3. Calcul du risque argile pour chaque ligne...")

def get_argile_risk(line, argile_zones):
    """Retourne le niveau de risque max pour une ligne"""
    try:
        buffer_geom = line['geometry'].buffer(0.05)  # Buffer de 50m
        intersecting = argile_zones[argile_zones.intersects(buffer_geom)]

        if len(intersecting) > 0:
            max_risk = intersecting['niveau'].max()
            return max_risk  # 2.0 = Moyen, 3.0 = Fort
        return 1.0  # Pas de risque
    except:
        return 1.0

rail_lines['argile_risk'] = [get_argile_risk(line, argile_large) for idx, line in rail_lines.iterrows()]

# 4. Classer le risque
print("\n4. Classification des lignes:")
risk_counts = rail_lines['argile_risk'].value_counts()
print(f"   Sans risque (niveau 1): {risk_counts.get(1.0, 0):,} lignes")
print(f"   Risque moyen (niveau 2): {risk_counts.get(2.0, 0):,} lignes")
print(f"   Risque fort (niveau 3): {risk_counts.get(3.0, 0):,} lignes")

# 5. Créer la carte
print("\n5. Génération de la carte...")
argile_rail_map = folium.Map(location=[46.603354, 1.888334], zoom_start=6, tiles='CartoDB positron')

# Ajouter les lignes SNCF colorées par risque
def style_rail_risk(feature):
    risk = feature['properties'].get('argile_risk', 1.0)
    if risk == 3.0:
        return {'color': 'red', 'weight': 3, 'opacity': 0.8}
    elif risk == 2.0:
        return {'color': 'orange', 'weight': 3, 'opacity': 0.8}
    elif risk == 1.0:
        return {'color': 'green', 'weight': 2, 'opacity': 0.8}

folium.GeoJson(
    rail_lines,
    style_function=style_rail_risk,
    tooltip=folium.GeoJsonTooltip(
        fields=['code_ligne', 'argile_risk'],
        aliases=['Ligne:', 'Risque argile:'],
        labels=True
    ),
    name='Réseau SNCF'
).add_to(argile_rail_map)

# Légende
legend_html = """
<div style="position: fixed; top: 50px; right: 50px; width: 220px; height: 120px;
     border:2px solid grey; z-index:9999; font-size:14px; background-color:white; padding: 12px;">
     <b>Risque Argile SNCF</b><br><hr>
     <i style="background:green; width:20px; height:3px; display:inline-block; border:1px solid grey;"></i> Sans risque<br>
     <i style="background:orange; width:20px; height:3px; display:inline-block; border:1px solid grey;"></i> Risque moyen<br>
     <i style="background:red; width:20px; height:3px; display:inline-block; border:1px solid grey;"></i> Risque fort<br>
</div>
"""

argile_rail_map.get_root().html.add_child(folium.Element(legend_html))

print("\n✓ Carte générée avec succès!")
print("   Le réseau SNCF est coloré selon le risque d'argile")

argile_rail_map